# Data cleaning

* Removal of full duplicit rows
* Removal of instances assigned to more categories at the same time
* Merge of perex and title columns into a new combined text column
* Remove the smallest categories
* Split data to train, validation and test splits with concrete seed

### Module imports

In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

import sys

In [2]:
sys.path.insert(0, str(Path.cwd().parent))

from src.config import RAW_DATA_PATH, CLEAN_DATA_PATH
from src.config import SEED, VAL_SIZE, TEST_SIZE

In [3]:
df = pd.read_csv(RAW_DATA_PATH)

In [4]:
original_row_count = df.shape[0]
print(f"We have loaded {original_row_count} rows")

We have loaded 111218 rows


In [5]:
df.head()

,category,rss_title,rss_perex
0,biatlon,"Krčmář dojel v hromadném závodě devátý, díky s...",Závod s hromadným startem v německém Oberhofu ...
1,biatlon,Česká vlajka byla v Pokljuce vidět i ve štafet...,Galerie
2,biatlon,Živě: Stíhací závod biatlonistek v Ruhpoldingu,"15. 1., 14:45"
3,fotbal,Bakoš dostal herdu do nosu a v zápase plném ka...,Slovenský útočník Marek Bakoš zařídil Plzni gó...
4,fotbal,"My moc chtěli a Plzeň moc nechtěla, zní z Ďolí...",Český fotbal hledá viníka skandálu odloženého ...


### Drop duplicate rows

In [6]:
df.drop_duplicates(inplace=True)
no_duplicates_row_count = df.shape[0]
print(f"We have dropped {original_row_count - no_duplicates_row_count} rows")

We have dropped 7185 rows


### Drop conflicting rows

In [7]:
conflict = df.groupby(["rss_title", "rss_perex"])["category"].transform("nunique") > 1
df = df[~conflict]

In [8]:
print(f"We have dropped {no_duplicates_row_count - df.shape[0]} conflicting rows")

We have dropped 27 conflicting rows


### Drop instances associated with categories that have too few samples

In [14]:
MIN_CLASS_COUNT = 10

counts = df["category"].value_counts()
small = counts[counts < MIN_CLASS_COUNT].index

print(f"MIN_CLASS_COUNT = {MIN_CLASS_COUNT}")
print(f"The classes that are too small are {list(small)}")

MIN_CLASS_COUNT = 10
The classes that are too small are []


In [15]:
before_small_drop = df.shape[0]
df = df[~df["category"].isin(small)].reset_index(drop=True)
print(f"We have dropped {before_small_drop - df.shape[0]} rows across {len(small)} class(es): {list(small)}")

We have dropped 0 rows across 0 class(es): []


### Build the text column

In [16]:
df["text"] = df["rss_title"] + " " + df["rss_perex"]
df = df.reset_index(drop=True)
df.head(3)

,category,rss_title,rss_perex,text
0,biatlon,"Krčmář dojel v hromadném závodě devátý, díky s...",Závod s hromadným startem v německém Oberhofu ...,"Krčmář dojel v hromadném závodě devátý, díky s..."
1,biatlon,Česká vlajka byla v Pokljuce vidět i ve štafet...,Galerie,Česká vlajka byla v Pokljuce vidět i ve štafet...
2,biatlon,Živě: Stíhací závod biatlonistek v Ruhpoldingu,"15. 1., 14:45",Živě: Stíhací závod biatlonistek v Ruhpoldingu...


# Split to train, val and test splits

Split to train + val AND test (combined)

In [17]:
val_test_size = VAL_SIZE + TEST_SIZE                
train_idx, val_test_idx = train_test_split(
    df.index, test_size=val_test_size, stratify=df["category"], random_state=SEED,
)

Split val AND test set to validation and testing sets that are separate

In [18]:
test_frac = TEST_SIZE / val_test_size           
val_idx, test_idx = train_test_split(
    val_test_idx, test_size=test_frac, stratify=df.loc[val_test_idx, "category"], random_state=SEED,
)

Assign to appropriate columns

In [19]:
df["split"] = "train"
df.loc[val_idx, "split"] = "val"
df.loc[test_idx, "split"] = "test"

In [20]:
print(df["split"].value_counts(normalize=True).round(3))

split
train    0.8
test     0.1
val      0.1
Name: proportion, dtype: float64


In [21]:
distribution = (
    df.groupby("split")["category"]
    .value_counts(normalize=True)
    .unstack(level=0)
    .mul(100)
    .round(2)
)
display(distribution.sort_values("train", ascending=False))

split,test,train,val
category,,,
fotbal,41.02,41.01,41.02
hokej,25.81,25.82,25.82
tenis,10.45,10.45,10.44
formule1,5.42,5.42,5.42
moto,4.48,4.48,4.48
atletika,2.22,2.22,2.21
basketbal,2.21,2.21,2.20
cyklistika,1.79,1.79,1.78
biatlon,1.41,1.41,1.41


### Export to intermediary file

In [ ]:
df.to_parquet(CLEAN_DATA_PATH)
print(f"Saved: {CLEAN_DATA_PATH}  ({len(df)} rows)")